### To download the report about gilts, go here: https://www.dmo.gov.uk/data/pdfdatareport?reportCode=D1A

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import pandas as pd
from datetime import datetime

### Load current gilts prices

In [ ]:
# -----------------------------
# Step 1: Load the web page
# -----------------------------
url = "https://www.hl.co.uk/shares/corporate-bonds-gilts/bond-prices/uk-gilts?column=maturity&order=asc"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
response = requests.get(url, headers=headers)

if response.status_code != 200:
    raise Exception(f"Failed to load page, status code: {response.status_code}")

html = response.text

# -----------------------------
# Step 2: Parse with BeautifulSoup
# -----------------------------
soup = BeautifulSoup(html, "html.parser")

# Find the table containing gilts
# The page uses <table> for the bond prices
table = soup.find("table")  # assuming first table is the correct one

# -----------------------------
# Step 3: Extract table rows
# -----------------------------
rows = table.find_all("tr")[1:]  # skip header row

data = []

for row in rows:
    cells = row.find_all("td")
    if len(cells) < 5:
        continue  # skip incomplete rows

    # --- Issuer ---
    issuer_tag = cells[0].find("a", class_="link-headline")
    if issuer_tag:
        issuer_text = issuer_tag.get_text(strip=True)
    else:
        issuer_text = ""

    # Extract coupon from issuer text (e.g., "Treasury 1.5% 22/07/2026")
    coupon_match = re.search(r"(\d+\.?\d*)%", issuer_text)
    coupon = float(coupon_match.group(1)) if coupon_match else None

    # Extract maturity date from issuer text
    maturity_match = re.search(r"(\d{2}/\d{2}/\d{4})", issuer_text)
    maturity = maturity_match.group(1) if maturity_match else ""

    # --- Price ---
    price_text = cells[2].get_text(strip=True).replace(",", "")
    try:
        price = float(price_text)
    except:
        price = None

    # Actions column (usually links/buttons)
    actions = cells[4].get_text(strip=True)

    data.append({
        "Issuer": issuer_text,
        "Coupon (%)": coupon,
        "Maturity": maturity,
        "Price": price,
        "Actions": actions
    })

# -----------------------------
# Step 4: Create DataFrame
# -----------------------------
gilts_df = pd.DataFrame(data)
print(gilts_df.head())

### Define helper functions

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------

def parse_date(date_str):
    """Parse a date string like '31 Jan 2028'"""
    return datetime.strptime(date_str.strip(), "%d %b %Y")

def accrued_interest(clean_price, coupon_rate, settlement_date, last_coupon_date, face_value=100):
    """
    Calculate accrued interest to add to clean price.
    """
    days_between_coupons = (settlement_date - last_coupon_date).days
    coupon_period_days = 182.5  # approx 6 months
    semi_coupon = coupon_rate / 2 * face_value
    return semi_coupon * (days_between_coupons / coupon_period_days)

def generate_coupon_dates(first_issue, redemption, dividend_dates):
    """
    Generate all coupon dates between first issue and redemption.
    dividend_dates: string like '22 Jan/Jul'
    """
    start_year = first_issue.year
    end_year = redemption.year
    schedule = []

    day_months = []
    for dm in dividend_dates.split('/'):
        day, mon = dm.split()
        day_months.append((int(day), mon))

    for y in range(start_year, end_year + 1):
        for day, mon in day_months:
            try:
                dt = datetime.strptime(f"{day} {mon} {y}", "%d %b %Y")
            except:
                continue
            if dt > first_issue and dt <= redemption:
                schedule.append(dt)
    return sorted(schedule)

def simplified_yield(dirty_price, coupons, face_value, years_to_maturity):
    """
    Compute after-tax annual yield assuming all cashflows are at maturity
    """
    total_inflows = sum(coupons) + face_value
    r = (total_inflows / dirty_price) ** (1 / years_to_maturity) - 1
    return r

### Example usage

In [ ]:
# -----------------------------
# Example input tables
# -----------------------------

# Market table (clean prices)
market_data = pd.DataFrame({
    "Issuer": ["Treasury 0.125% 31/01/2028"],
    "Coupon (%)": [0.125],
    "Maturity": ["31 Jan 2028"],
    "Price": [92.490]  # clean price
})

# Gilt details table
details_data = pd.DataFrame({
    "Conventional Gilts": ["Treasury 0.125% 31/01/2028"],
    "ISIN Code": ["GB00BMBL1G81"],
    "Redemption Date": ["31 Jan 2028"],
    "First Issue Date": ["22 Jan 2023"],
    "Dividend Dates": ["22 Jan/Jul"],
    "Ex-dividend Date": ["22 Jan 2028"],
    "Amount in Issue (£ million nominal)": ["1000"]
})

# Settlement date
settlement = datetime(2025, 8, 31)

# -----------------------------
# Compute simplified after-tax yield for each gilt
# -----------------------------

results = []

for _, row in market_data.iterrows():
    clean_price = row["Price"]
    coupon_rate = row["Coupon (%)"]
    maturity = parse_date(row["Maturity"])

    # Match gilt details
    details = details_data[details_data["Conventional Gilts"] == row["Issuer"]].iloc[0]
    first_issue = parse_date(details["First Issue Date"])
    dividend_dates = details["Dividend Dates"]

    # Coupon schedule
    coupon_schedule = generate_coupon_dates(first_issue, maturity, dividend_dates)
    last_coupon = max([d for d in coupon_schedule if d <= settlement])

    # Dirty price = clean + accrued interest
    dirty_price = clean_price + accrued_interest(clean_price, coupon_rate, settlement, last_coupon)

    # Number of semi-annual coupons remaining (after settlement)
    remaining_coupons = [c for c in coupon_schedule if c > settlement]
    semi_coupon_value = coupon_rate / 2 * 100  # per £100 face
    coupons_after_tax = [semi_coupon_value * (1 - 0.45) for _ in remaining_coupons]

    # Fractional years to maturity
    years_to_maturity = (maturity - settlement).days / 365

    # Simplified annual yield
    r = simplified_yield(dirty_price, coupons_after_tax, 100, years_to_maturity)

    results.append({
        "Gilt": row["Issuer"],
        "Dirty Price": round(dirty_price, 3),
        "After-tax annual yield": round(r * 100, 4)
    })

# -----------------------------
# Show results
# -----------------------------
results_df = pd.DataFrame(results)
print(results_df)